# Forecast-period ensemble map metrics

Maps of ensemble verification metrics (bias, RMSE, spread, CRPS, ...) for the atmospheric hindcasts (`fc`) against their references, one figure per case, variable and metric. Global scalar metrics and rank histograms are in `02_compute_and_plot_scalar_metrics.ipynb`.

Cases: **S0** hindcasts start from the Dec 2011 analyses and cover Jan-Feb 2012; **S1** hindcasts start from the May 2012 analyses and cover Jun-Jul 2012. Each case compares hindcasts initialized from CTRL, CAPT and EAM-DART states. Model fields are the monthly means over both forecast months.

## 1. Settings

In [ ]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
workflow_name = "fcst_atm_model_evaluation"

component = "atm"
run_segment = "fc"
# Forecast cases: season -> experiments compared together (same period).
cases = {
    "S0": ("CTRL10-S0", "CAPT10-S0", "DART10-S0", "DART20-S0", "DART40-S0"),  # 201201-201202
    "S1": ("CTRL10-S1", "CAPT10-S1", "DART40-S1"),  # 201206-201207
}
variables = ("TS", "TREFHT", "PSL", "TMQ", "PRECT", "FLUT")
map_metrics = ("bias", "rmse", "spread", "crps")
n_bootstrap = 0  # > 0 adds 95% ensemble-bootstrap CIs (hatching where bias is significant)

# Model input averaged over the forecast months. None -> monthly h0 means
# ("clim") for forecasts; "daily" or "6hourly" applies to every variable.
model_source = None
lat_range = (-90, 90)
lon_range = (-180, 180)
max_members = None  # e.g. 2 for a quick test; caches record the ensemble size
random_seed = 42
# Caches are rebuilt automatically when they are missing, older than any
# model/reference input, or were built with other settings; force_compute
# rebuilds them regardless.
force_compute = False
show = True
FIGURE_DPI = 200

## 2. Project setup

In [ ]:
import sys
import matplotlib.pyplot as plt

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.model_evaluation import MAP_METRICS, evaluate_cases, plot_evaluation_maps

data_dir, figure_dir = workflow_output_dirs(
    workflow_name, output_root=DIAGNOSTIC_OUTPUT_ROOT
)
unknown = set(map_metrics) - set(MAP_METRICS)
if unknown:
    raise ValueError(f"Unknown map metrics {sorted(unknown)}; choose from {MAP_METRICS}")

## 3. Compute or reuse map metrics

In [ ]:
results, registries, failures = evaluate_cases(
    "map", cases, variables, data_root=INPUT_DATA_ROOT, cache_dir=data_dir,
    component=component, run_segment=run_segment, model_source=model_source,
    n_bootstrap=n_bootstrap, random_seed=random_seed,
    max_members=max_members, lat_range=lat_range, lon_range=lon_range,
    force_compute=force_compute,
)
if failures:
    print(f"\n{len(failures)} case(s) failed and are skipped below:")
    for season, variable, experiment, error in failures:
        print(f"  {season} {variable} {experiment}: {error}")

## 4. Plot, save, and show

In [ ]:
for season, by_variable in results.items():
    registry = registries[season]
    for variable, datasets in by_variable.items():
        period = registry[next(iter(datasets))]["period"]
        for metric in map_metrics:
            fig = plot_evaluation_maps(
                datasets, metric, variable,
                label_map={name: registry[name]["label"] for name in datasets},
                title=f"{variable}: {metric} ({season} {run_segment}, {period})",
            )
            output = figure_dir / f"{variable}_{metric}_{season}_{run_segment}.png"
            fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
            print(f"[FIGURE] {output}")
            if not show:
                plt.close(fig)

## Notes: variables and map metrics

Atmosphere variables: `FLUT` (CERES-OAFlux), `PRECT` (GPCP), `TMQ`, `TS`, `TREFHT`, and `PSL` (ERA5).

| Metric | Definition |
|---|---|
| `mean`, `reference_mean` | time-mean ensemble mean / reference |
| `bias` | ensemble mean minus reference, time mean |
| `bias_stddev` | temporal standard deviation of the ensemble-mean error |
| `rmse` | RMSE of individual members over members and time |
| `spread` | ensemble standard deviation, time mean |
| `crps` | continuous ranked probability score, time mean |
| `coverage` | fraction of times the reference lies inside the ensemble envelope |

Caches: `<experiment>_<variable>_<period>_map_metrics.nc` in the workflow data directory.